# 用户兴趣演化-序列建模
无论是二阶交叉的FM、AFM，还是高阶交叉的DCN、xDeepFM，它们的核心目标都是从一个静态的特征集合中挖掘出有价值的信息。然而，这些模型普遍存在一个共同的局限：它们大多将用户的历史行为看作一个无序的“物品袋”（a bag of items），如同用户的兴趣是一个静态的表示。

但用户的兴趣不是静止的，而是具有明显的时序性和动态演化特点。一个用户先浏览“鼠标”再浏览“显示器”，与先浏览“小说”再浏览“显示器”，这两个行为序列背后指向的购买意图截然不同。前者可能是一位正在组装电脑的数码爱好者，而后者可能只是在工作之余的随性浏览。传统的特征交叉模型难以捕捉这种蕴含在行为顺序中的、随时间变化的意图。

因此，本节我们将转换视角，不再将用户历史看作一堆静态特征的集合，而是将其视为一个动态的序列。我们将聚焦于如何对用户的行为序列进行建模，从这个序列中挖掘出用户动态、演化的兴趣。接下来，我们将介绍工业界在序列建模方向上的三个代表性模型：DIN、DIEN和DSIN，看看它们是如何解决这个核心挑战的。
- DIN
- DIEN
- DISN
- SIM

序列建模的三个关键模型：DIN通过注意力机制解决用户兴趣多样性问题，DIEN进一步建模兴趣的时序演化过程，DSIN则引入会话概念进行分层建模。这些模型体现了序列建模的核心思想：动态性（根据任务调整兴趣表示）、序列性（利用时间顺序信息）和聚焦性（针对任务筛选相关信息）。随着技术发展，未来的序列建模方法将结合更多先进技术来更好地理解用户动态需求。

### 参考
[SIM: 基于搜索的超长行为序列上的用户兴趣建模](https://zhuanlan.zhihu.com/p/370951728)

## 深度兴趣网络（Deep Interest Network, DIN）
在大型电商平台中，用户的兴趣是多样的。一个用户可能在一段时间内，既关注数码产品，又浏览运动装备，还会购买生活用品。在传统的深度学习模型（即Embedding&MLP范式）中，通常的做法是将用户所有的历史行为（如点击过的商品ID）对应的Embedding向量通过**池化（Pooling）**操作，压缩成一个**固定长度**的向量来代表该用户。

这个固定长度的用户向量，很快就成为了表达用户多样兴趣的瓶颈。想象一下，无论系统准备向这个用户推荐“跑鞋”还是“手机”，代表他的都是同一个向量。这个向量试图“一视同仁”地蕴含该用户所有的兴趣点，这不仅非常困难，而且在面对具体推荐任务时显得不够聚焦。为了增强表达能力而粗暴地增加向量维度，又会带来参数量爆炸和过拟合的风险。

**DIN的核心思想：局部激活 (Local Activation)**

**深度兴趣网络（Deep Interest Network, DIN） (Zhou et al., 2018) **的提出者们发现，用户的某一次具体点击行为，通常只由其历史兴趣中的一部分所“激活”。当向一位数码爱好者推荐“机械键盘”时，真正起决定性作用的，很可能是他最近浏览“游戏鼠标”和“显卡”的行为，而不是他上个月购买的“跑鞋”。

基于此，DIN提出了一个观点：**用户的兴趣表示不应该是固定的，而应是根据当前的候选广告（Candidate Ad）不同而动态变化的。**
![alt text](resource/DIN.png)

### 技术实现：注意力机制

为了实现“局部激活”这一思想，DIN 在模型中引入了一个关键模块——**局部激活单元（Local Activation Unit）**，其本质就是**注意力机制**。如上图右侧所示，DIN 不再像基准模型（图3.3.1 左）那样对所有历史行为的 Embedding 进行简单的池化，而是进行了一次“加权求和”。

这个权重（即注意力分数）的计算，体现了 DIN 的核心思想。具体来说，对于一个给定的用户 $ U $ 和候选广告 $ A $，用户的兴趣表示向量 $ \mathbf{v}_U(A) $ 是这样计算的：

$$
\mathbf{v}_U(A) = f(\mathbf{v}_A, \mathbf{e}_1, \mathbf{e}_2, \ldots, \mathbf{e}_H) = \sum_{j=1}^{H} a(\mathbf{e}_j, \mathbf{v}_A)\mathbf{e}_j = \sum_{j=1}^{H} w_j \mathbf{e}_j \tag{3.3.1}
$$

其中：

- $ \mathbf{e}_1, \mathbf{e}_2, \ldots, \mathbf{e}_H $ 是用户 $ U $ 的历史行为 Embedding 向量列表。
- $ \mathbf{v}_A $ 是候选广告 $ A $ 的 Embedding 向量。
- $ a(\mathbf{e}_j, \mathbf{v}_A) $ 是一个激活单元（通常是一个小型前馈神经网络），它接收历史行为 $ \mathbf{e}_j $ 和候选广告 $ \mathbf{v}_A $ 作为输入，输出一个权重 $ w_j $。这个权重就代表了历史行为 $ \mathbf{e}_j $ 在面对广告 $ \mathbf{v}_A $ 时的“相关性”或“注意力得分”。

- $ a(\mathbf{e}_j, \mathbf{v}_A) $是把$\mathbf{e}_i和\mathbf{v}_A$以及$\mathbf{e}_i，\mathbf{v}_A$的element wise差值向量合并起来作为输入，然后喂给全连接层，最后得出权重，
- 在得出weight后用dropout来加强泛化性

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LocalActivateUnit(nn.Module):
    def __init__(self,emb_dim, dropout_rate=0.1):
        super(LocalActivateUnit, self).__init__()
        
        self.emb_dim = emb_dim
        
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim*4, emb_dim),
            nn.ReLU(),
            nn.Linear(emb_dim, 1)
        )
        
        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self,target_item, user_behaviors, mask=None):
        # target_item:(batch,emb_dim)
        # user_behaviors:(batch,seq_len, emb_dim)
        # mask:(batch,seq_len)
        print(f'target_item shape:{target_item.shape}')
        print(f'user_behaviors shape:{user_behaviors.size()}')
        # print(f'mask shape:{mask.size()}')
        
        _, seq_len, emb_dim = user_behaviors.size()
        # 显式修正target_item维度与user_behaviors相同，少用广播机制
        target_item = target_item.unsqueeze(1).expand(-1,seq_len, -1)
        # tensor.unsqueeze(add_dim), add_dim为新增的维度，默认为1
        # tensor.expand(-1,new_dim, -1), -1表示维持不变， new_dim表示扩展后的新维度
        # tensor.expand与tensor.view与tensor.reshape的区别
        
        # 这里DIN用线性映射+简单特征交叉的方式，替代了传统attention的内积，表达能力更强
        # 特征交叉，interactions: (batch_size,seq_len, emb_dim * 4)
        interaction = torch.concat([target_item, user_behaviors, target_item-user_behaviors, target_item*user_behaviors], dim=-1)
        # dim=-1，对最后维度进行拼接
        print(f'interaction shape:{interaction.shape}')
        
        # 通过两个线性层映射变换
        # att_scores:(batch,seq_len, emb_dim)
        att_scores = self.ffn(interaction).squeeze(-1)
        print(f'att_scores shape:{att_scores.shape}')
        
        if mask is not None:
            att_scores = att_scores.masked_fill(
                mask == 0, float('-inf')
            )
        
        # att_weight:(batch,seq_len)
        att_weight = F.softmax(att_scores, dim=-1)
        att_weight = self.dropout(att_weight)
        # att_weight:(batch,seq_len, emb_dim)
        att_weight = att_weight.unsqueeze(-1).expand(-1,-1,emb_dim)
        print(f'att_weight shape:{att_weight.shape}')
        
        #user_behaviors:(batch, emb_dim)
        user_interest = torch.sum(att_weight * user_behaviors, dim=1)
        print(f'user_interest shape:{user_interest.shape}')
        
        return user_interest

# test
def test():
    batch = 2
    seq_len = 3
    emb_dim = 5
    target = torch.randn([batch, emb_dim])
    user_behaviors = torch.randn([batch,seq_len,emb_dim])
    net = LocalActivateUnit(emb_dim)
    user_interest = net(target, user_behaviors)
# Test
if __name__ == "__main__":
    test()

target_item shape:torch.Size([2, 5])
user_behaviors shape:torch.Size([2, 3, 5])
interaction shape:torch.Size([2, 3, 20])
att_scores shape:torch.Size([2, 3])
att_weight shape:torch.Size([2, 3, 5])
user_interest shape:torch.Size([2, 5])


参考：[推荐系统中的注意力机制——阿里深度兴趣网络（DIN）](https://zhuanlan.zhihu.com/p/51623339)

## SIM
### Overall
目前在DIN和DIEN上，使用了注意力机制，让候选item，attend到用户的行为序列上。但这样有一个问题，就是由于latency的原因，用户序列不能太长，所以只能attend到用户最近的行为序列上。

而我们知道，一个用户在淘宝上的行为序列是很长的，比如23%的用户最近5个月会点击超过1000个产品。如果只是使用用户最近的行为序列，那么信息会有丢弃，导致效果变差。

那么，用什么办法解决呢？

注意到在DIN中的使用attention的动机是用户的兴趣是多元的，为了触发用户的某个兴趣点，才做了attention，那么attention得到的权重中必然是那些跟候选item相近的产品才会权重比较大。比如，候选商品是包，那么用户行为序列中和包相关才会得到比较大的权重。

因此，可以考虑使用search的方法，在用户行为序列中进行初步筛选，得到相关的item再去做attention。也因此，本文的标题是Search-based user interest modeling。

### 两步策略
由上一节中，我们知道，需要先做初步筛选，然后再做类似DIN的模型处理。如下图所示：
![alt text](resource/SIM.png)

上图左侧是第一个阶段，即初步筛选。这里分为两个策略，第一个是soft search, 第二个是hard search。 soft search是用候选item的embedding去和用户行为序列中的每一项的embedding去做点积，然后去top-K。这里可以使用的是一些高效方法是ALSH和MIPS的，都是已有的方法，咱们在基于Delaunay图的快速最大内积搜索算法中介绍过MIPS的方法。hard search是利用item的一些元信息，比如商品类别，在用户的行为序列中进行选择，这个方法是无参的。两种策略公式如下:
$$
r_i = 
\begin{cases}
\text{Sign}(C_i = C_a), & \text{hard-search} \\
(W_b \mathbf{e}_i) \odot (W_a \mathbf{e}_a)^T, & \text{soft-search}
\end{cases}
$$

实验表明，hard search虽然结果会稍微差一些，但是会快很多。

**soft search本质上是离线训练一个item-item的双塔模型，样本构造选用同一用户购买了的brand作为正，没有购买的作为负样本，选用1：10，用focal loss计算。
结构式将item输入到同一个FFN，然后将输出的emb求点积，为相似度**

无论是soft search还是hard search，在获取序列后都是用前面所提到的DIN或者DIEN进行处理，得到点击率预估的值。

特别注意一点，那就是虽然在serving的时候分了两步，但是在训练的时候，是同时训练的。之所以这样，是因为第一步和第二步所需要的用户序列是不一样的，第一步是用户的全部序列，第二步是选择后的序列，所以，想要在第一步中的序列的embedding效果好，需要在第一步中添加一个Auxulary loss。

第一步被称之为**General Search Unit， 简称GSU**。第二步被称为**Exact Search Unit，简称ESU**。

GSU是将lastN转变为TopK

GSU做完后，就得到了用户行为序列中相对于候选item的序列子集，然后对于序列子集中的每个item，提取了两种特征，第一种是item的embedding，第二种是item相对于候选item的时间差，即时间信息。这两种信息拼接起来，得到的embedding去做DIN。

### 补充-SIM的user_behavior需要记录交互时间，并且把时间按照3/7/15/30/60/90/90以上，进行编码，然后输入到embding层中
- 用户与某个 LastN 物品的交互时刻距今为 $\delta$。
- 对 $\delta$ 做离散化，再做 embedding，变成向量 $\mathbf{d}$。
- 把两个向量做 concatenation，表征一个 LastN 物品。
  - 向量 $\mathbf{x}$ 是物品 embedding。
  - 向量 $\mathbf{d}$ 是时间的 embedding。

## 深度兴趣演化网络（Deep Interest Evolution Network, DIEN）
DIN成功地捕捉了用户兴趣的“多样性”和“局部激活”特性，**但它仍然存在一个局限：它将用户的历史行为看作是一个无序的集合，忽略了行为之间的时序依赖关系**。用户的兴趣不仅是多样的，更是在**持续演化**的。

为了解决这个问题，**深度兴趣演化网络（Deep Interest Evolution Network, DIEN） (Zhou et al., 2019)** 被提出。DIEN认为，我们不仅要关注哪些历史兴趣是相关的，更要理解这些兴趣是如何一步步演化至今的。

DIEN的核心思想是，直接对原始、显性的行为序列建模是不够的。行为只是表象，我们更应该关注行为背后那个**潜在的、抽象的 “兴趣”状态**，并对这个兴趣状态的演化过程进行建模。为此，DIEN设计了一个两阶段结构，如上图所示。
![alt text](resource/DIEN.png)

## 深度会话兴趣网络（Deep Session Interest Network, DSIN）
从DIN到DIEN，我们看到了模型对用户兴趣的理解从“静态相关”走向了“动态演化”。然而，它们都将用户的行为看作一条连续的序列。但现实中，用户的行为模式更多是间断性的。用户通常在一个**会话（Session）** 内拥有一个明确且集中的意图，而在**不同会话**之间，兴趣点可能发生巨大转变。

一个用户可能在一个会话里集中浏览各种裤子，而在下一个会话则专注于戒指。这种**会话内同质、会话间异质**的现象非常普遍。如果直接用一个RNN模型处理这种“断层”明显的长序列，模型需要花费很大力气去学习这种兴趣的突变，效果并不理想。

**深度会话兴趣网络（Deep Session Interest Network, DSIN） (Feng et al., 2019)** 基于这一观察，提出我们应该将**“会话”**作为分析用户行为的基本单元，并采用一种分层的思想来建模。
![alt text](resource/DSIN.png)

### DSIN的技术实现：分层建模

DSIN 的架构如上图所示，其建模过程可以清晰地分为以下几个层次：

1. **会话划分层 (Session Division Layer)**：这是模型的第一步，也是 DSIN 的基础。它根据行为发生的时间间隔（例如，如果两个行为间隔超过 30 分钟），将原始的、连续的用户行为长序列 $ \mathbf{S} $，切分成多个独立的**会话短序列** $ \mathbf{Q} = [\mathbf{Q}_1, \mathbf{Q}_2, \ldots, \mathbf{Q}_K] $。

2. **会话兴趣提取层 (Session Interest Extractor Layer)**：这一层的目标是为每一个会话 $ \mathbf{Q}_k $ 提取出一个核心的兴趣向量。DSIN 认为，一个会话内的行为虽然意图集中，但彼此之间的重要性也不同。因此，它没有使用简单的池化，而是采用了**自注意力机制（Self-Attention）**（与 Transformer 的核心思想一致）。自注意力网络能够捕捉该会话内部所有行为之间的内在关联，并聚合最重要的信息，最终为每个会话 $ \mathbf{Q}_k $ 生成一个浓缩的兴趣向量 $ \mathbf{I}_k $。

3. **会话兴趣交互层 (Session Interest Interacting Layer)**：经过上一步，我们得到了一个更高层次的序列——**会话兴趣向量的序列** $ \mathbf{I}_1, \mathbf{I}_2, \ldots, \mathbf{I}_K $。这个序列反映了用户兴趣在更长的时间尺度上的演变。DSIN 使用一个**双向长短时记忆网络（Bi-LSTM）**来对这个会话序列进行建模，从而捕捉不同会话之间的演进和依赖关系。Bi-LSTM 的输出是一个包含了上下文信息的会话兴趣序列 $ [\mathbf{H}_1, \mathbf{H}_2, \ldots, \mathbf{H}_K] $。

4. **会话兴趣激活层 (Session Interest Activating Layer)**：最后一步与 DIN 的思想一脉相承。模型会根据当前的**候选广告** $ \mathbf{X}_I $，使用注意力机制来计算每个会话兴趣的重要性，并进行加权求和，得到最终的用户兴趣表示。DSIN 分别对会话兴趣提取层和交互层的输出都进行了激活：

$$
\mathbf{U}^I = \sum_{k=1}^{K} a_k^I \mathbf{I}_k \quad \text{和} \quad \mathbf{U}^H = \sum_{k=1}^{K} a_k^H \mathbf{H}_k \tag{3.3.6}
$$

其中，$ a_k^I $ 和 $ a_k^H $ 是根据候选广告计算出的注意力权重。最终，将这两个激活后的向量 $ \mathbf{U}^I $ 和 $ \mathbf{U}^H $ 拼接，得到用户的最终兴趣表示。

---

DSIN 通过引入“会话”这一更符合用户实际行为模式的中间单元，将复杂的长序列建模问题分解为“**会话内信息聚合**”（通过自注意力）和“**会话间信息传递**”（通过 Bi-LSTM）两个更清晰的子问题。这种分层建模思想，使得模型能够对用户兴趣进行更精细的刻画。